# LeakLock Kawaii Image Check

A compact notebook GUI for uploading one image, running the LeakLock layered risk pipeline, and saving the JSON/CSV results.

Current pipeline flow:

- YOLOv8 detection layer
- Routing layer
- Face age risk layer
- OCR extraction layer
- OCR risk evaluation layer
- License plate fixed-risk layer

Notes:

- The notebook uses the trained YOLO weights configured in `PipelineConfig`.
- Face-age estimation tries ONNX first, then a Transformers/PyTorch age classifier, then DeepFace if installed.
- OCR risk evaluation currently uses the baseline rule-based classifier.
- Results are automatically saved to `analysis_results/json` and summarized in `analysis_results/summary.csv`.


In [28]:
# Optional: run this once if the environment still needs the notebook dependencies.
# %pip install -r ../requirements.txt
# If you install or upgrade notebook packages here, restart the kernel afterward.

In [2]:
from pathlib import Path
from datetime import datetime
import csv
import importlib
import json
import sys

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
from PIL import Image as PILImage

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force-refresh the local package so notebook runs use the code currently on disk.
for module_name in [name for name in list(sys.modules) if name == 'leaklock' or name.startswith('leaklock.')]:
    sys.modules.pop(module_name, None)

import leaklock
import leaklock.config
import leaklock.pipeline
import leaklock.layers.face_age
import leaklock.services.age_estimators

importlib.reload(leaklock.config)
importlib.reload(leaklock.services.age_estimators)
importlib.reload(leaklock.layers.face_age)
importlib.reload(leaklock.pipeline)
importlib.reload(leaklock)

from leaklock import LeakLockPipeline, PipelineConfig

UPLOAD_DIR = REPO_ROOT / 'uploaded_images'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = REPO_ROOT / 'analysis_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
JSON_RESULTS_DIR = RESULTS_DIR / 'json'
JSON_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OCR_RESULTS_DIR = RESULTS_DIR / 'ocr_text'
OCR_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV_PATH = RESULTS_DIR / 'summary.csv'

config = PipelineConfig(repo_root=REPO_ROOT)
pipeline = LeakLockPipeline(config=config)

age_estimator = pipeline._face_layer._estimator
estimator_stack = (
    ' -> '.join(estimator.__class__.__name__ for estimator in age_estimator._estimators)
    if hasattr(age_estimator, '_estimators')
    else age_estimator.__class__.__name__
)
ocr_provider = pipeline._ocr_layer._provider
ocr_provider_name = ocr_provider.__class__.__name__ if ocr_provider is not None else 'lazy; loads on first OCR detection'
SETUP_DETAILS = {
    'repo_root': REPO_ROOT,
    'upload_dir': UPLOAD_DIR,
    'results_dir': RESULTS_DIR,
    'ocr_results_dir': OCR_RESULTS_DIR,
    'summary_csv_path': SUMMARY_CSV_PATH,
    'yolo_weights': config.yolo_weights_path,
    'age_estimator_class': age_estimator.__class__.__name__,
    'age_estimator_stack': estimator_stack,
    'onnx_age_model_repo': getattr(config, 'hf_age_model_repo_id', 'not configured'),
    'transformers_age_model': getattr(config, 'hf_age_classifier_model_id', 'not configured'),
    'ocr_provider_class': ocr_provider_name,
}

In [3]:
def iter_uploaded_files(file_upload_value):
    if isinstance(file_upload_value, dict):
        for file_name, payload in file_upload_value.items():
            content = payload['content'] if isinstance(payload, dict) else payload.content
            yield file_name, content
        return

    for item in file_upload_value:
        if isinstance(item, dict):
            file_name = item.get('name', 'uploaded_image')
            content = item.get('content', b'')
        else:
            file_name = getattr(item, 'name', 'uploaded_image')
            content = getattr(item, 'content', b'')
        yield file_name, content


def save_uploaded_file(file_name, content):
    file_path = UPLOAD_DIR / file_name
    file_path.write_bytes(content)
    return file_path


def make_result_basename(image_path):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    return f"{image_path.stem}_{timestamp}"


def append_summary_row(result, image_path, json_path):
    file_exists = SUMMARY_CSV_PATH.exists()
    with SUMMARY_CSV_PATH.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=['image_name', 'image_path', 'overall_risk_percent', 'detections_count', 'json_result_path']
        )
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'image_name': image_path.name,
            'image_path': str(image_path),
            'overall_risk_percent': result.overall_risk_percent,
            'detections_count': len(result.analyses),
            'json_result_path': str(json_path),
        })


def save_ocr_text_file(result, image_path):
    ocr_text_blocks = []
    for analysis in result.analyses:
        if getattr(analysis, 'route', None) != 'ocr_extraction_layer':
            continue
        evidence = analysis.risk.evidence if hasattr(analysis.risk, 'evidence') else {}
        ocr_info = evidence.get('ocr', {}) if isinstance(evidence, dict) else {}
        text = (ocr_info.get('text') or '').strip() if isinstance(ocr_info, dict) else ''
        if text:
            ocr_text_blocks.append(f"[{analysis.detection.class_name}]\n{text}")

    if not ocr_text_blocks:
        return None

    txt_path = OCR_RESULTS_DIR / f"{make_result_basename(image_path)}.txt"
    txt_path.write_text("\n\n".join(ocr_text_blocks), encoding='utf-8')
    return txt_path


def save_result_files(result, image_path):
    base_name = make_result_basename(image_path)
    json_path = JSON_RESULTS_DIR / f'{base_name}.json'
    json_path.write_text(json.dumps(result.to_dict(), indent=2), encoding='utf-8')
    append_summary_row(result, image_path, json_path)
    ocr_text_path = save_ocr_text_file(result, image_path)
    return json_path, ocr_text_path


def analyze_and_display(image_path):
    display(Markdown(f'## {image_path.name}'))
    display(PILImage.open(image_path))
    result = pipeline.analyze_image(image_path)
    json_path, ocr_text_path = save_result_files(result, image_path)
    print(json.dumps(result.to_dict(), indent=2))
    print(f'JSON result saved to: {json_path}')
    if ocr_text_path is not None:
        print(f'OCR text saved to: {ocr_text_path}')
    print(f'CSV summary saved to: {SUMMARY_CSV_PATH}')
    return result

## LeakLock Studio


In [4]:
import hashlib
import html
import io
import traceback

from IPython.display import HTML as DisplayHTML

for widget_name in (
    "leaklock_gui",
    "kawaii_uploader",
    "analyze_button",
    "reset_button",
    "preview_name_html",
    "preview_image",
    "preview_placeholder_html",
    "preview_shell",
    "result_output",
    "status_html",
):
    old_widget = globals().get(widget_name)
    if hasattr(old_widget, "close"):
        try:
            old_widget.close()
        except Exception:
            pass

KAWAII_CSS = """
<style>
.leaklock-kawaii {
    max-width: 980px;
    margin: 10px 0 24px;
    padding: 22px;
    border: 1px solid #ffd7e5;
    border-radius: 16px;
    background: linear-gradient(135deg, #fff7fb 0%, #f7fbff 48%, #fffdf4 100%);
    box-shadow: 0 16px 42px rgba(181, 103, 139, 0.16);
    font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}
.leaklock-kawaii .hero {
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 18px;
    margin-bottom: 16px;
}
.leaklock-kawaii .title {
    margin: 0;
    color: #5b3148;
    font-size: 30px;
    line-height: 1.08;
    font-weight: 800;
    letter-spacing: 0;
}
.leaklock-kawaii .subtitle {
    margin: 6px 0 0;
    color: #7a6270;
    font-size: 14px;
}
.leaklock-kawaii .pill-row {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    justify-content: flex-end;
}
.leaklock-kawaii .pill {
    border: 1px solid rgba(216, 123, 159, 0.22);
    border-radius: 999px;
    padding: 6px 10px;
    color: #694157;
    background: rgba(255, 255, 255, 0.72);
    font-size: 12px;
    font-weight: 700;
}
.leaklock-kawaii .workspace {
    display: grid;
    grid-template-columns: minmax(260px, 0.88fr) minmax(320px, 1.12fr);
    gap: 16px;
}
.leaklock-kawaii .panel {
    border: 1px solid rgba(216, 123, 159, 0.18);
    border-radius: 14px;
    background: rgba(255, 255, 255, 0.78);
    padding: 14px;
    box-shadow: 0 8px 22px rgba(91, 49, 72, 0.07);
}
.leaklock-kawaii .panel-title {
    color: #5b3148;
    font-size: 14px;
    font-weight: 800;
    margin-bottom: 10px;
}
.leaklock-kawaii .hint {
    color: #8a7480;
    font-size: 12px;
    margin-top: 8px;
}
.leaklock-kawaii .preview-frame {
    overflow: hidden;
    border-radius: 12px;
    border: 1px dashed #f3bdd1;
    min-height: 220px;
    background: #fffafd;
    display: flex;
    align-items: center;
    justify-content: center;
}
.leaklock-kawaii .empty-preview {
    color: #b4879a;
    font-weight: 700;
    text-align: center;
    padding: 42px 18px;
}
.leaklock-kawaii .preview-name {
    color: #5b3148;
    font-weight: 800;
    margin: 10px 0 8px;
    word-break: break-word;
}
.leaklock-kawaii .status {
    border-radius: 12px;
    padding: 10px 12px;
    color: #684358;
    background: #fff1f7;
    border: 1px solid #ffd5e4;
    font-size: 13px;
    font-weight: 700;
}
.leaklock-kawaii .status.busy { background: #f0f7ff; border-color: #cfe6ff; color: #315579; }
.leaklock-kawaii .status.good { background: #f0fff7; border-color: #bfe8d0; color: #2e6849; }
.leaklock-kawaii .status.bad { background: #fff2f1; border-color: #ffc4bf; color: #9b4037; }
.leaklock-kawaii .result-grid {
    display: grid;
    grid-template-columns: minmax(140px, 0.7fr) minmax(180px, 1.3fr);
    gap: 12px;
    margin-bottom: 12px;
}
.leaklock-kawaii .risk-card {
    border-radius: 14px;
    padding: 16px;
    color: #4e3443;
    background: #f8fbff;
    border: 1px solid #dbeafe;
}
.leaklock-kawaii .risk-card.medium { background: #fff7df; border-color: #ffe2a8; }
.leaklock-kawaii .risk-card.high { background: #fff0f3; border-color: #ffc5d2; }
.leaklock-kawaii .risk-card .label {
    font-size: 12px;
    text-transform: uppercase;
    color: #77606c;
    font-weight: 800;
}
.leaklock-kawaii .risk-card .score {
    font-size: 42px;
    font-weight: 900;
    line-height: 1;
    margin-top: 7px;
}
.leaklock-kawaii .mini-stat {
    border-radius: 12px;
    background: #ffffff;
    border: 1px solid #f0dbe4;
    padding: 10px 12px;
    margin-bottom: 8px;
}
.leaklock-kawaii .mini-stat b {
    color: #5b3148;
}
.leaklock-kawaii .detection-card {
    border-radius: 12px;
    border: 1px solid #eed8e2;
    background: #fffefe;
    padding: 12px;
    margin-top: 10px;
}
.leaklock-kawaii .detection-head {
    display: flex;
    justify-content: space-between;
    gap: 10px;
    align-items: center;
    color: #5b3148;
    font-weight: 800;
}
.leaklock-kawaii .detection-meta {
    color: #806b76;
    font-size: 12px;
    margin-top: 4px;
}
.leaklock-kawaii .reason {
    color: #593849;
    margin-top: 8px;
    font-size: 13px;
}
.leaklock-kawaii code {
    color: #674057;
    background: #fff3f8;
    border-radius: 6px;
    padding: 2px 5px;
}
.leaklock-kawaii pre {
    max-height: 280px;
    overflow: auto;
    background: #2b2230;
    color: #fff7fb;
    border-radius: 10px;
    padding: 12px;
    font-size: 12px;
}
.leaklock-button-row {
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    align-items: center;
}
@media (max-width: 760px) {
    .leaklock-kawaii .hero,
    .leaklock-kawaii .workspace,
    .leaklock-kawaii .result-grid {
        display: block;
    }
    .leaklock-kawaii .pill-row { justify-content: flex-start; margin-top: 12px; }
    .leaklock-kawaii .panel { margin-top: 12px; }
}
</style>
"""
display(DisplayHTML(KAWAII_CSS))

_selected_upload = {"name": None, "content": None, "hash": None}
_last_processed_hash = None
_last_result = None
_is_analyzing = False
_suppress_upload_change = False


def _html(text):
    return html.escape(str(text), quote=True)


def get_first_uploaded_file(value):
    if isinstance(value, dict) and value:
        name, info = next(iter(value.items()))
        content = info["content"] if isinstance(info, dict) else info.content
        if hasattr(content, "tobytes"):
            content = content.tobytes()
        return (info.get("name", name) if isinstance(info, dict) else name), bytes(content)

    if isinstance(value, (tuple, list)) and len(value) > 0:
        info = value[0]
        name = info.get("name", "uploaded_image") if isinstance(info, dict) else getattr(info, "name", "uploaded_image")
        content = info["content"] if isinstance(info, dict) else getattr(info, "content", b"")
        if hasattr(content, "tobytes"):
            content = content.tobytes()
        return name, bytes(content)

    return None, None


def _risk_tone(percent):
    if percent >= 60:
        return "high"
    if percent >= 30:
        return "medium"
    return "low"


def _set_status(message, tone="idle"):
    status_html.value = f'<div class="status {tone}">{_html(message)}</div>'


def _preview_empty():
    preview_name_html.value = ""
    preview_image.value = b""
    preview_image.layout.display = "none"
    preview_placeholder_html.layout.display = None
    preview_placeholder_html.value = '<div class="empty-preview">No image selected</div>'


def _clear_browser_upload_value():
    global _suppress_upload_change
    _suppress_upload_change = True
    try:
        kawaii_uploader.value = ()
    except Exception:
        pass
    if hasattr(kawaii_uploader, "_counter"):
        kawaii_uploader._counter = 0
    _suppress_upload_change = False


def _image_widget_format(file_name):
    suffix = Path(file_name).suffix.lower().lstrip('.')
    if suffix in {'jpg', 'jpeg'}:
        return 'jpeg'
    if suffix in {'png', 'gif', 'webp'}:
        return suffix
    return 'png'


def _render_preview(file_name, content):
    preview_name_html.value = f'<div class="preview-name">{_html(file_name)}</div>'
    try:
        PILImage.open(io.BytesIO(content)).verify()
        preview_placeholder_html.layout.display = "none"
        preview_image.format = _image_widget_format(file_name)
        preview_image.value = content
        preview_image.layout.display = None
    except Exception:
        preview_image.value = b""
        preview_image.layout.display = "none"
        preview_placeholder_html.layout.display = None
        preview_placeholder_html.value = '<div class="empty-preview">Preview unavailable</div>'


def _result_summary_html(result, image_path, json_path, ocr_text_path):
    percent = result.overall_risk_percent
    tone = _risk_tone(percent)
    json_text = _html(json_path)
    ocr_text = _html(ocr_text_path) if ocr_text_path is not None else "None"
    return f"""
    <div class="result-grid">
        <div class="risk-card {tone}">
            <div class="label">Overall risk</div>
            <div class="score">{percent}%</div>
        </div>
        <div>
            <div class="mini-stat"><b>Image:</b> {_html(image_path.name)}</div>
            <div class="mini-stat"><b>Detections:</b> {len(result.analyses)}</div>
            <div class="mini-stat"><b>JSON:</b> <code>{json_text}</code></div>
            <div class="mini-stat"><b>OCR text:</b> <code>{ocr_text}</code></div>
        </div>
    </div>
    """


def _detection_cards_html(result):
    if not result.analyses:
        return '<div class="detection-card"><div class="detection-head">No sensitive objects detected</div></div>'

    cards = []
    for index, analysis in enumerate(result.analyses, start=1):
        detection = analysis.detection
        risk = analysis.risk
        evidence = risk.evidence or {}
        age_estimate = evidence.get("age_estimate", {}) if isinstance(evidence, dict) else {}
        age_line = ""
        if isinstance(age_estimate, dict) and age_estimate.get("age_years") is not None:
            confidence = age_estimate.get("confidence")
            confidence_text = f" - confidence {float(confidence):.2f}" if confidence is not None else ""
            age_line = f'<div class="detection-meta">Age estimate: <b>{_html(age_estimate.get("age_years"))}</b>{confidence_text}</div>'
        elif isinstance(age_estimate, dict) and age_estimate:
            age_line = f'<div class="detection-meta">Age estimate unavailable: {_html(age_estimate.get("details", "no details"))}</div>'

        cards.append(f"""
        <div class="detection-card">
            <div class="detection-head">
                <span>{index}. {_html(detection.class_name.title())}</span>
                <span>{risk.risk_percent}%</span>
            </div>
            <div class="detection-meta">route: {_html(analysis.route)} - detection confidence {detection.confidence:.2f}</div>
            {age_line}
            <div class="reason">{_html(risk.reason)}</div>
        </div>
        """)
    return "".join(cards)


def _render_result(result, image_path, json_path, ocr_text_path):
    raw_json = json.dumps(result.to_dict(), indent=2)
    summary = widgets.HTML(_result_summary_html(result, image_path, json_path, ocr_text_path))
    detections = widgets.HTML(_detection_cards_html(result))
    raw = widgets.HTML(f"<pre>{_html(raw_json)}</pre>")
    accordion = widgets.Accordion(children=[raw])
    accordion.set_title(0, "Raw JSON")
    with result_output:
        clear_output(wait=True)
        display(widgets.VBox([summary, detections, accordion]))


def _reset_upload(*_):
    global _selected_upload, _last_processed_hash, _is_analyzing
    _selected_upload = {"name": None, "content": None, "hash": None}
    _last_processed_hash = None
    _is_analyzing = False
    analyze_button.disabled = True
    _clear_browser_upload_value()
    _preview_empty()
    with result_output:
        clear_output(wait=True)
    _set_status("Ready", "idle")


def _on_upload_change(change):
    if _suppress_upload_change:
        return

    file_name, content = get_first_uploaded_file(change.get("new"))
    if content is None:
        return

    file_hash = hashlib.sha1(content).hexdigest()
    if file_hash == _selected_upload.get("hash"):
        _clear_browser_upload_value()
        return

    _selected_upload["name"] = file_name
    _selected_upload["content"] = content
    _selected_upload["hash"] = file_hash
    analyze_button.disabled = False
    _render_preview(file_name, content)
    _set_status(f"Selected {file_name}", "good")
    _clear_browser_upload_value()


def _run_analysis(*_):
    global _last_processed_hash, _last_result, _is_analyzing
    if _is_analyzing:
        return

    file_name = _selected_upload.get("name")
    content = _selected_upload.get("content")
    if not file_name or content is None:
        _set_status("Choose an image first", "bad")
        return

    file_hash = _selected_upload.get("hash") or hashlib.sha1(content).hexdigest()
    if file_hash == _last_processed_hash:
        _set_status("This image is already analyzed. Reset or choose another image.", "idle")
        analyze_button.disabled = True
        return

    _is_analyzing = True
    analyze_button.disabled = True
    _set_status("Analyzing image", "busy")
    with result_output:
        clear_output(wait=True)
        display(DisplayHTML('<div class="status busy">Running LeakLock pipeline...</div>'))

    try:
        image_path = save_uploaded_file(file_name, content)
        result = pipeline.analyze_image(image_path)
        json_path, ocr_text_path = save_result_files(result, image_path)
        _last_result = result
        _last_processed_hash = file_hash
        _render_result(result, image_path, json_path, ocr_text_path)
        _set_status(f"Analysis complete - risk {result.overall_risk_percent}%", "good")
    except Exception as exc:
        with result_output:
            clear_output(wait=True)
            display(DisplayHTML(
                '<div class="status bad">Analysis failed</div>'
                f'<pre>{_html(traceback.format_exc())}</pre>'
            ))
        _set_status(f"Failed: {exc}", "bad")
    finally:
        _is_analyzing = False
        analyze_button.disabled = True


header_html = widgets.HTML("""
<div class="hero">
    <div>
        <h1 class="title">LeakLock Image Check</h1>
        <div class="subtitle">Pastel privacy scan for faces, IDs, documents, and plates.</div>
    </div>
    <div class="pill-row">
        <span class="pill">YOLOv8</span>
        <span class="pill">Face age</span>
        <span class="pill">OCR risk</span>
    </div>
</div>
""")

kawaii_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Choose image",
    icon="folder-open",
    layout=widgets.Layout(width="170px"),
)
analyze_button = widgets.Button(
    description="Analyze",
    icon="search",
    disabled=True,
    layout=widgets.Layout(width="132px"),
)
reset_button = widgets.Button(
    description="Reset",
    icon="refresh",
    layout=widgets.Layout(width="112px"),
)

kawaii_uploader.add_class("leaklock-upload")
analyze_button.add_class("leaklock-action")
reset_button.add_class("leaklock-reset")

status_html = widgets.HTML()
preview_name_html = widgets.HTML()
preview_image = widgets.Image(
    value=b"",
    format="png",
    layout=widgets.Layout(width="100%", max_height="320px", object_fit="contain", display="none"),
)
preview_placeholder_html = widgets.HTML()
preview_shell = widgets.VBox([preview_placeholder_html, preview_image])
preview_shell.add_class("preview-frame")
result_output = widgets.Output()

controls = widgets.HBox([kawaii_uploader, analyze_button, reset_button])
controls.add_class("leaklock-button-row")
left_panel = widgets.VBox([
    widgets.HTML('<div class="panel-title">Image</div>'),
    controls,
    status_html,
    preview_name_html,
    preview_shell,
])
left_panel.add_class("panel")

right_panel = widgets.VBox([
    widgets.HTML('<div class="panel-title">Result</div>'),
    result_output,
])
right_panel.add_class("panel")

workspace = widgets.HBox([left_panel, right_panel])
workspace.add_class("workspace")

leaklock_gui = widgets.VBox([header_html, workspace])
leaklock_gui.add_class("leaklock-kawaii")

kawaii_uploader.observe(_on_upload_change, names="value")
analyze_button.on_click(_run_analysis)
reset_button.on_click(_reset_upload)

_preview_empty()
_set_status("Ready", "idle")
display(leaklock_gui)
